# BiteBuddy Recommendation Benchmark

This notebook runs offline ranking benchmarks on the Food.com dataset and compares five baseline strategies:

- `random`
- `popularity`
- `rating`
- `content_profile`
- `hybrid`

Evaluation protocol:

- positive interactions are ratings `>= 4`
- chronological per-user split
- last positive interaction per user is held out as test
- each user is evaluated against `199` sampled negatives plus `1` ground-truth positive
- metrics reported at `K=10`: `HitRate`, `Recall`, `NDCG`, `MRR`, and `CatalogCoverage`


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

repo_root = Path.cwd()
python_exe = Path(sys.executable)
eval_script = repo_root / 'backend' / 'scripts' / 'evaluate_recommenders.py'
output_path = repo_root / 'backend' / 'data' / 'processed' / 'recommender_eval.json'

python_exe, eval_script, output_path

(WindowsPath('C:/Users/suriy/miniconda3/envs/bitebuddy/python.exe'),
 WindowsPath('D:/BiteBuddy-Food-Recipe-Agent/backend/scripts/evaluate_recommenders.py'),
 WindowsPath('D:/BiteBuddy-Food-Recipe-Agent/backend/data/processed/recommender_eval.json'))

In [2]:
cmd = [
    str(python_exe),
    str(eval_script),
    '--sample-users', '500',
    '--negatives-per-user', '199',
    '--output', str(output_path),
]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(result.stdout)

{
  "config": {
    "sample_users": 500,
    "negatives_per_user": 199,
    "min_positive_interactions": 5,
    "positive_rating_threshold": 4,
    "seed": 42
  },
  "dataset": {
    "recipes": 231637,
    "interactions": 1132367,
    "positive_interactions": 1003724
  },
  "strategies": {
    "random": {
      "users_evaluated": 500,
      "HitRate@10": 0.064,
      "Recall@10": 0.064,
      "NDCG@10": 0.0271,
      "MRR@10": 0.0162,
      "CatalogCoverage@10": 0.0214
    },
    "popularity": {
      "users_evaluated": 500,
      "HitRate@10": 0.308,
      "Recall@10": 0.308,
      "NDCG@10": 0.2311,
      "MRR@10": 0.2074,
      "CatalogCoverage@10": 0.0186
    },
    "rating": {
      "users_evaluated": 500,
      "HitRate@10": 0.202,
      "Recall@10": 0.202,
      "NDCG@10": 0.0991,
      "MRR@10": 0.0678,
      "CatalogCoverage@10": 0.0182
    },
    "content_profile": {
      "users_evaluated": 500,
      "HitRate@10": 0.134,
      "Recall@10": 0.134,
      "NDCG@10": 0.0621,
  

In [3]:
report = json.loads(output_path.read_text(encoding='utf-8'))
report['config'], report['dataset']

({'sample_users': 500,
  'negatives_per_user': 199,
  'min_positive_interactions': 5,
  'positive_rating_threshold': 4,
  'seed': 42},
 {'recipes': 231637,
  'interactions': 1132367,
  'positive_interactions': 1003724})

In [4]:
metrics = pd.DataFrame(report['strategies']).T
metrics = metrics[['users_evaluated', 'HitRate@10', 'Recall@10', 'NDCG@10', 'MRR@10', 'CatalogCoverage@10']]
metrics = metrics.sort_values(['HitRate@10', 'NDCG@10'], ascending=False)
metrics

,users_evaluated,HitRate@10,Recall@10,NDCG@10,MRR@10,CatalogCoverage@10
hybrid,500.0,0.322,0.322,0.2388,0.2130,0.0187
popularity,500.0,0.308,0.308,0.2311,0.2074,0.0186
rating,500.0,0.202,0.202,0.0991,0.0678,0.0182
content_profile,500.0,0.134,0.134,0.0621,0.0410,0.0207
random,500.0,0.064,0.064,0.0271,0.0162,0.0214


In [5]:
best = metrics.index[0]
print(f'Best offline strategy in this benchmark: {best}')
print('\nLift over random baseline:')
random_metrics = metrics.loc['random']
for col in ['HitRate@10', 'NDCG@10', 'MRR@10']:
    lift = metrics.loc[best, col] - random_metrics[col]
    print(f'{col}: {metrics.loc[best, col]:.4f} vs random {random_metrics[col]:.4f}  (delta {lift:+.4f})')

Best offline strategy in this benchmark: hybrid

Lift over random baseline:
HitRate@10: 0.3220 vs random 0.0640  (delta +0.2580)
NDCG@10: 0.2388 vs random 0.0271  (delta +0.2117)
MRR@10: 0.2130 vs random 0.0162  (delta +0.1968)


## Interpretation

Expected outcome for this project:

- `random` is only a sanity baseline
- `popularity` is usually strong because recipe interactions are head-heavy
- `rating` is useful but weaker alone
- `content_profile` is limited because Food.com user preference signals are sparse and noisy
- `hybrid` should be the best current baseline because it combines user-profile content overlap with collaborative popularity and rating priors

For BiteBuddy, this supports a production strategy of:

1. hard constraint filtering
2. hybrid candidate generation
3. reranking with collaborative and content signals
4. LLM-based explanation and follow-up refinement
